# Three-block worked example — power supply + MCU + CAN transceiver

A small automotive-style ECU: a linear LDO produces a 5 V rail from the 12 V battery; the MCU runs the application and CAN stack off that rail; the CAN transceiver drives the bus, also off the same rail. Three blocks, one DAG, real cross-block dependencies.

**The pipeline:**

```
          ┌─ scenarios.toml (vbat, ambient_temp by corner)
          ├─ modes.toml      (off / sleep / active / diagnostic)
          └─ project/requirements.py  (RAIL_5V, MCU_T_J_MAX, ...)
                  │
                  ▼
          Project.load() ──▶ project.run([three blocks of modules],
                                          return_all_computed=True)
                  │                                  │
                  ▼                                  ▼
          results: dict[str, Quantity]          test_results (run_verifications)
                  │                                  │
                  └────────► block_report_html ◄────┘
                  └────────► project_report_html ◄──┘
                  └────────► quantity_to_html (with provenance) 
```

**Cross-block wiring (design doc 6.7 — Contracts as cut-points):**

- `power_supply` publishes Contract **`rail_5v`** (the 5 V envelope; 4.75–5.25 V).
- `can_transceiver` and `mcu` both *consume* `rail_5v` from their analysis functions.
- They publish their own draw Contracts (**`can_5v_draw`**, **`mcu_5v_draw`**).
- `power_supply` sums those Contracts into its total-load math; the framework's cycle-cut rule ensures no block reaches past another's Contracts into its internals.
- `power_supply`'s Contract has `assumed_inputs={can_5v_draw, mcu_5v_draw}` — bounds the PSU assumed when sizing its thermal design. Step 8b validates the assumption.

## 1. Load the project, import the three blocks

In [1]:
import sys
from pathlib import Path

# Make `blocks/`, `project/` (here) and `components/` (one dir up) importable.
sys.path.insert(0, str(Path.cwd()))
sys.path.insert(0, str(Path.cwd().parent))

from framework import Project, requirements

# Trigger requirement registration.
from project.requirements import RAIL_5V, OPERATING_TEMP, MCU_T_J_MAX, PSU_T_J_MAX

# Three blocks — each a self-contained subpackage.
from blocks.can_transceiver import (
    leaves as can_leaves,
    analysis as can_analysis,
    contracts as can_contracts,
    verifications as can_verifications,
)
from blocks.mcu import (
    leaves as mcu_leaves,
    analysis as mcu_analysis,
    contracts as mcu_contracts,
    verifications as mcu_verifications,
)
from blocks.power_supply import (
    leaves as psu_leaves,
    analysis as psu_analysis,
    contracts as psu_contracts,
    verifications as psu_verifications,
)

# DAG_MODULES feed Hamilton (leaves + analysis + contracts).
# VERIFICATION_MODULES feed run_verifications (module scan for @verification_test).
# Keeping them separate prevents Hamilton from seeing verification-test
# functions as DAG nodes — they take a `ctx` arg, which would otherwise
# leak as an extra input node.
DAG_MODULES = [
    can_leaves, can_analysis, can_contracts,
    mcu_leaves, mcu_analysis, mcu_contracts,
    psu_leaves, psu_analysis, psu_contracts,
]
VERIFICATION_MODULES = [can_verifications, mcu_verifications, psu_verifications]

project = Project.load(
    scenarios=Path.cwd() / "project" / "scenarios.toml",
    modes=Path.cwd() / "project" / "modes.toml",
)

print(f"Project: {len(project.scenarios)} scenarios, "
      f"{len(project.modes)} modes, "
      f"{len(requirements.list_all())} requirements")
print(f"Scenarios: {project.scenarios.names()}")
print(f"Modes:     {project.modes.names()}")

Project: 3 scenarios, 4 modes, 12 requirements
Scenarios: ['nominal', 'cold_low_vin', 'hot_high_vin']
Modes:     ['off', 'sleep', 'active', 'diagnostic']


## 2. The DAG

`Project.list_nodes(modules)` enumerates every function Hamilton wires together. Each block contributes a handful of nodes (block-prefixed for namespace hygiene — Hamilton uses a single flat namespace by parameter name).

In [2]:
import inspect
from framework import get_contract_meta

print("DAG nodes:", project.list_nodes(DAG_MODULES))
print()

for label, mods in (
    ("power_supply", [psu_leaves, psu_analysis, psu_contracts]),
    ("mcu",          [mcu_leaves, mcu_analysis, mcu_contracts]),
    ("can_transceiver", [can_leaves, can_analysis, can_contracts]),
):
    print(f"=== blocks/{label}/ ===")
    for module in mods:
        for fn_name in dir(module):
            if fn_name.startswith("_"):
                continue
            fn = getattr(module, fn_name)
            if not callable(fn) or getattr(fn, "__module__", None) != module.__name__:
                continue
            sig = inspect.signature(fn)
            marker = "  [CONTRACT]" if get_contract_meta(fn) else ""
            print(f"  def {fn_name}({', '.join(sig.parameters)}) -> Quantity{marker}")
    print()

DAG nodes: ['ambient_temp', 'can_5v_draw', 'can_i_supply', 'can_power', 'can_r_theta_ja', 'can_t_j', 'can_t_j_max', 'can_thermal_rise', 'mcu_5v_draw', 'mcu_i_supply', 'mcu_power', 'mcu_r_theta_ja', 'mcu_t_j', 'mcu_t_j_max', 'mcu_thermal_rise', 'psu_power_dissipation', 'psu_r_theta_ja', 'psu_t_j', 'psu_t_j_max', 'psu_thermal_rise', 'psu_total_5v_load', 'psu_v_drop', 'psu_v_out_actual', 'rail_5v', 'vbat']

=== blocks/power_supply/ ===
  def psu_r_theta_ja() -> Quantity
  def psu_t_j_max() -> Quantity
  def psu_v_out_actual() -> Quantity
  def psu_power_dissipation(psu_v_drop, psu_total_5v_load) -> Quantity
  def psu_t_j(ambient_temp, psu_thermal_rise) -> Quantity
  def psu_thermal_rise(psu_power_dissipation, psu_r_theta_ja) -> Quantity
  def psu_total_5v_load(can_5v_draw, mcu_5v_draw) -> Quantity
  def psu_v_drop(vbat, psu_v_out_actual) -> Quantity
  def rail_5v() -> Quantity  [CONTRACT]

=== blocks/mcu/ ===
  def mcu_i_supply() -> Quantity
  def mcu_r_theta_ja() -> Quantity
  def mcu_t_

## 3. Run the DAG + verifications

`return_all_computed=True` returns every node Hamilton evaluated, including Contracts and their `compares_to` actuals auto-augmented for the consistency checks — the design-review renderers downstream need the full surface. The verification runner discovers `@verification_test` functions across all three blocks via a single module scan.

In [3]:
from framework import run_verifications

TARGETS = [
    # Headline values per block — the report renderers automatically pull
    # in Contracts + compares_to + assumed_inputs via return_all_computed.
    "can_t_j", "can_t_j_max", "can_power",
    "mcu_t_j", "mcu_t_j_max", "mcu_power",
    "psu_t_j", "psu_t_j_max", "psu_power_dissipation", "psu_total_5v_load",
]

results = project.run(
    modules=DAG_MODULES,
    targets=TARGETS,
    return_all_computed=True,
)

test_results = run_verifications(VERIFICATION_MODULES, results)
print(f"Computed nodes ({len(results)}): {sorted(results)}")
print()
print(f"Verification tests ({len(test_results)}):")
for name, r in test_results.items():
    status = "PASS" if r.passed else "FAIL"
    print(f"  [{status}] {name}")

Computed nodes (16): ['can_5v_draw', 'can_i_supply', 'can_power', 'can_t_j', 'can_t_j_max', 'mcu_5v_draw', 'mcu_i_supply', 'mcu_power', 'mcu_t_j', 'mcu_t_j_max', 'psu_power_dissipation', 'psu_t_j', 'psu_t_j_max', 'psu_total_5v_load', 'psu_v_out_actual', 'rail_5v']

Verification tests (6):
  [PASS] CAN: i_supply stays within published can_5v_draw contract
  [FAIL] CAN: t_j stays below block-derated max
  [PASS] MCU: i_supply stays within published mcu_5v_draw contract
  [FAIL] MCU: t_j stays below block-derated max
  [PASS] PSU: LDO actual output within published rail_5v envelope
  [FAIL] PSU: LDO t_j stays below block-derated max


## 4. PSU total load and dissipation — where the cross-block math happens

`psu_total_5v_load = can_5v_draw + mcu_5v_draw` is the cleanest illustration of cross-block composition: two blocks publish mode-keyed Contracts and the PSU consumes their sum via Hamilton's parameter-name wiring. The LDO dissipation then folds in the scenario-dependent V_in to surface a worst case at hot ambient + high battery.

In [4]:
from framework.units import mA

total_load = results["psu_total_5v_load"].to(mA)
p_diss = results["psu_power_dissipation"]

scen_names = project.scenarios.names()
mode_names = project.modes.names()

def cell(val, fmt="{:6.2f}"):
    if isinstance(val, tuple):
        lo, hi = val
        return f"{fmt.format(lo)} – {fmt.format(hi)}"
    return fmt.format(val)

print("Total 5V load (mA, mode-keyed):")
for m in mode_names:
    print(f"  {m:12s} {cell(total_load.at(mode=m), '{:5.2f}')} mA")
print()
print("PSU LDO dissipation (mW, scenario × mode):")
print(f"  {'mode':12s} | " + " | ".join(f"{s:^18s}" for s in scen_names))
print("  " + "-" * (14 + 21 * len(scen_names)))
for m in mode_names:
    cells = []
    for s in scen_names:
        v = p_diss.at(mode=m, scenario=s)
        cells.append(cell(v, '{:7.1f}'))
    print(f"  {m:12s} | " + " | ".join(f"{c:^18s}" for c in cells) + "  mW")

Total 5V load (mA, mode-keyed):
  off           0.00 mA
  sleep         0.00 –  0.03 mA
  active        0.00 – 95.00 mA
  diagnostic    0.00 – 130.00 mA

PSU LDO dissipation (mW, scenario × mode):
  mode         |      nominal       |    cold_low_vin    |    hot_high_vin   
  -----------------------------------------------------------------------------
  off          |          0.0       |          0.0       |          0.0        mW
  sleep        |     0.0 –     0.2  |     0.0 –     0.1  |     0.0 –     0.3   mW
  active       |     0.0 –   669.8  |     0.0 –   384.8  |     0.0 –  1049.8   mW
  diagnostic   |     0.0 –   916.5  |     0.0 –   526.5  |     0.0 –  1436.5   mW


## 5. Junction temperatures — three blocks, one scenario × mode grid

The headline thermal numbers for each block. The LDO is hottest by a wide margin at hot_high_vin (linear-regulator dissipation explodes when V_in is high *and* I_load is heavy), the CAN transceiver is next, and the MCU comfortably stays in margin except at the worst diagnostic-mode corner.

In [5]:
def cell_temp(val):
    if isinstance(val, tuple):
        lo, hi = val
        return f"{lo:5.1f} – {hi:5.1f}"
    return f"{val:5.1f}"

for label, node in (("CAN  (limit 125 °C)", "can_t_j"),
                    ("MCU  (limit 90 °C)",  "mcu_t_j"),
                    ("PSU  (limit 125 °C)", "psu_t_j")):
    q = results[node]
    print(f"\n{label} — t_j in °C:")
    print(f"  {'mode':12s} | " + " | ".join(f"{s:^15s}" for s in scen_names))
    print("  " + "-" * (14 + 18 * len(scen_names)))
    for m in mode_names:
        cells = [cell_temp(q.at(mode=m, scenario=s)) for s in scen_names]
        print(f"  {m:12s} | " + " | ".join(f"{c:^15s}" for c in cells))


CAN  (limit 125 °C) — t_j in °C:
  mode         |     nominal     |  cold_low_vin   |  hot_high_vin  
  --------------------------------------------------------------------
  off          |       25.0      |      -40.0      |       85.0     
  sleep        |   25.0 –  25.0  |  -40.0 – -40.0  |   85.0 –  85.0 
  active       |   50.6 –  65.9  |  -14.4 –   0.9  |  110.6 – 125.9 
  diagnostic   |   64.9 –  81.7  |   -0.1 –  16.7  |  124.9 – 141.7 

MCU  (limit 90 °C) — t_j in °C:
  mode         |     nominal     |  cold_low_vin   |  hot_high_vin  
  --------------------------------------------------------------------
  off          |       25.0      |      -40.0      |       85.0     
  sleep        |   25.0 –  25.0  |  -40.0 – -40.0  |   85.0 –  85.0 
  active       |   26.4 –  29.7  |  -38.6 – -35.3  |   86.4 –  89.7 
  diagnostic   |   28.4 –  32.9  |  -36.6 – -32.1  |   88.4 –  92.9 

PSU  (limit 125 °C) — t_j in °C:
  mode         |     nominal     |  cold_low_vin   |  hot_high_vin 

## 6. Verification results — rendered

The same `test_results` dict feeds the HTML table and the PR-comment markdown — one source of truth, two surfaces. Three thermal verifications fail at `hot_high_vin` (the design-margin alarm bell); three contract-bound verifications pass.

In [6]:
from framework import results_to_html_table, results_to_pr_comment
from IPython.display import HTML, Markdown, display

display(HTML(results_to_html_table(test_results)))

print("--- PR comment preview ---")
display(Markdown(results_to_pr_comment(test_results, title="System verification")))

Verdict,Severity,Test,Failed at
FAIL,critical,CAN: t_j stays below block-derated max,"scenario='hot_high_vin', mode='active'; scenario='hot_high_vin', mode='diagnostic'"
FAIL,critical,MCU: t_j stays below block-derated max,"scenario='hot_high_vin', mode='diagnostic'"
FAIL,critical,PSU: LDO t_j stays below block-derated max,"scenario='hot_high_vin', mode='active'; scenario='hot_high_vin', mode='diagnostic'"
PASS,critical,CAN: i_supply stays within published can_5v_draw contract,—
PASS,critical,MCU: i_supply stays within published mcu_5v_draw contract,—
PASS,critical,PSU: LDO actual output within published rail_5v envelope,—


--- PR comment preview ---


## System verification
**3 pass**, **3 fail**, **0 warn**, **0 info**

<details><summary>FAIL — CAN: t_j stays below block-derated max</summary>

- Severity: `critical`
- Failed at: scenario='hot_high_vin', mode='active'; scenario='hot_high_vin', mode='diagnostic'
- Evidence:
  - `can_t_j`
  - `can_t_j_max`

</details>

<details><summary>FAIL — MCU: t_j stays below block-derated max</summary>

- Severity: `critical`
- Failed at: scenario='hot_high_vin', mode='diagnostic'
- Evidence:
  - `mcu_t_j`
  - `mcu_t_j_max`

</details>

<details><summary>FAIL — PSU: LDO t_j stays below block-derated max</summary>

- Severity: `critical`
- Failed at: scenario='hot_high_vin', mode='active'; scenario='hot_high_vin', mode='diagnostic'
- Evidence:
  - `psu_t_j`
  - `psu_t_j_max`

</details>

## 7. Per-block design-review report

`block_report_html` renders one block's contracts (declared vs actual per mode, with status badges), assumed-input checks (cross-block), verifications filtered to that block's tests, every block-owned Quantity with a collapsible provenance chain, and a forward walk of the DAG showing which other blocks consume this block's contracts.

In [7]:
from framework import display_block_report

display_block_report(
    project,
    modules=DAG_MODULES,
    results=results,
    block="power_supply",
    test_results=test_results,
)

In [8]:
display_block_report(
    project,
    modules=DAG_MODULES,
    results=results,
    block="mcu",
    test_results=test_results,
)

In [9]:
display_block_report(
    project,
    modules=DAG_MODULES,
    results=results,
    block="can_transceiver",
    test_results=test_results,
)

## 8. Project-wide design-review report

Header counts, all-blocks verification roll-up, every block's section in one page, and a requirement-coverage matrix mapping each Jama ID to its contracts and verification verdicts. This is the artifact a design reviewer should read first; the per-block sections in §7 are for drilling in.

In [10]:
from framework import display_project_report

display_project_report(
    project,
    modules=DAG_MODULES,
    results=results,
    test_results=test_results,
)

## 9. Provenance — "where did this number come from?"

The PSU's `psu_t_j` at `hot_high_vin / diagnostic` is the headline failure. Walking its provenance chain shows every node that contributed, with hashes — including the cross-block jump from PSU's `psu_total_5v_load` back into CAN's and MCU's published Contracts.

In [11]:
from framework import display_quantity

psu_t_j = results["psu_t_j"]
print("PSU t_j provenance:")
print(psu_t_j.provenance.chain_summary())

print("\n--- Same quantity, rendered with provenance collapsible ---\n")
display_quantity(psu_t_j, name="PSU junction temperature (psu_t_j)")

PSU t_j provenance:
- psu_t_j  [blocks.power_supply.analysis::psu_t_j]  hash=11ede93975
  - ambient_temp  [input]  hash=30b4521299
  - psu_thermal_rise  [blocks.power_supply.analysis::psu_thermal_rise]  hash=4ba455edd2
    - psu_power_dissipation  [blocks.power_supply.analysis::psu_power_dissipation]  hash=e9a6fa368b
    - psu_r_theta_ja  [blocks.power_supply.leaves::psu_r_theta_ja]  hash=f412bd6405
      - psu_v_drop  [blocks.power_supply.analysis::psu_v_drop]  hash=e0aad44be7
      - psu_total_5v_load  [blocks.power_supply.analysis::psu_total_5v_load]  hash=b9095b7247
        - vbat  [input]  hash=2c244e1b16
        - psu_v_out_actual  [blocks.power_supply.leaves::psu_v_out_actual]  hash=977f692175
        - can_5v_draw  [blocks.can_transceiver.contracts::can_5v_draw]  hash=110d250184
        - mcu_5v_draw  [blocks.mcu.contracts::mcu_5v_draw]  hash=62d436e96d

--- Same quantity, rendered with provenance collapsible ---



Scenario,Mode,Value
cold_low_vin,off,-40 °C
hot_high_vin,off,85 °C
nominal,off,25 °C
cold_low_vin,sleep,-40 – -39.99 °C
hot_high_vin,sleep,85 – 85.02 °C
nominal,sleep,25 – 25.01 °C
cold_low_vin,active,-40 – -20.76 °C
hot_high_vin,active,85 – 137.5 °C
nominal,active,25 – 58.49 °C
cold_low_vin,diagnostic,-40 – -13.68 °C


## What this exercises

- **Block layout** (design doc 6.6): three independent subpackages, each `{leaves, analysis, contracts, verifications}`.
- **Contracts as cut-points** (6.7): PSU consumes published Contracts of CAN + MCU; CAN + MCU consume PSU's `rail_5v`. The cycle detector enforces no inter-block reach past Contracts.
- **Run-time contract consistency** (step 8): three Contracts × three `compares_to` actuals — all consistent on this run.
- **Cross-block assumed-input validation** (step 8b): PSU's Contract assumes upper bounds on CAN + MCU draws; the framework verifies those bounds at run time against the published Contract values.
- **Verification tests** (step 9a / 9b): three CRITICAL thermal checks (all failing at `hot_high_vin` — designed-to-fail to show how the framework surfaces design issues) + three contract-bound current checks (all passing).
- **Output channels** (step 9c): the same `test_results` dict renders to HTML table, PR-comment markdown, and Jama-shape JSON.
- **Provenance chain traversal** (step 10): the PSU's headline thermal failure walks back through both cross-block Contracts to the original component leaves.
- **Design-review reports** (step 12): per-block + project-wide HTML reports, with contracts tables (declared vs actual, status badges), verifications filtered per block, key Quantities with collapsible chains, and the cross-block consumer walk.